# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Melih-Yilmaz06/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1: High CTR strongly correlates with sustained search rankings.**
*Methodology Question:* Does the label (sustained search rankings) originate from a time window that strictly follows the feature window (CTR)? If a 90-day aggregate CTR overlaps with the 30-day ranking evaluation period, could this overlapping window introduce leakage? We should ensure features are strictly historical compared to the label.

**Finding 2: Content decline is highly predictable using historical engagement metrics.**
*Methodology Question:* What is the exact source of the ground-truth label for "content decline"? If the label is derived from `trend_direction` or `trend_pct`, are those same columns inadvertently included as features? Removing any label-derived features is critical to prevent the model from simply memorizing a tautology.

In [ ]:
print("Methodology questions drafted successfully.")


## 2. My model under an honest split (before/after)

Here we demonstrate the impact of data leakage from a random split versus an honest split.
When we use a random split, the model might memorize specific client patterns, artificially inflating performance. By switching to `GroupKFold` on the `client_id`, we enforce an honest evaluation: testing the model on clients it has never seen during training.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Load data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Generate target label if it doesn't exist explicitly in raw data
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = (df['trend_pct'] < 0).astype(int)

# 2. Prepare features and label
leaky_cols = ['trend_pct', 'trend_direction', 'is_declining_label', 'content_id']
df['has_word_count'] = df['word_count'].notnull().astype(int)
df['word_count'] = df['word_count'].fillna(df['word_count'].median())

features = [c for c in df.columns if c not in leaky_cols and pd.api.types.is_numeric_dtype(df[c])]
X = df[features].fillna(0) # naive fill for remaining numerics
y = df['is_declining_label']
groups = df['client_id']

# 3. Random Split (Dishonest)
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.2, random_state=42)
clf_rand = RandomForestClassifier(random_state=42, max_depth=5)
clf_rand.fit(X_train_rand, y_train_rand)
score_rand = roc_auc_score(y_test_rand, clf_rand.predict_proba(X_test_rand)[:, 1])

# 4. Grouped Split (Honest)
gkf = GroupKFold(n_splits=5)
scores_honest = []
for train_idx, test_idx in gkf.split(X, y, groups):
    X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
    y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
    
    clf_g = RandomForestClassifier(random_state=42, max_depth=5)
    clf_g.fit(X_train_g, y_train_g)
    scores_honest.append(roc_auc_score(y_test_g, clf_g.predict_proba(X_test_g)[:, 1]))

score_honest = np.mean(scores_honest)

print(f"Random Split AUC (Dishonest): {score_rand:.4f}")
print(f"Grouped Split AUC (Honest):   {score_honest:.4f}")
print(f"Gap (Memorization penalty):   {score_rand - score_honest:.4f}")


## 3. Leakage audit

In this audit, we actively hunt for data leakage. The most critical rule from our schema rules is that `is_declining_label` is derived from `trend_direction` and `trend_pct`. Therefore, these columns must be strictly excluded from the feature set.

Additionally, we identify failure cases (False Positives/Negatives) to understand where our honest model struggles.

In [ ]:
from IPython.display import display

# Validate leakage removal
assert 'trend_pct' not in features, "Leakage Alert: trend_pct is in features!"
assert 'trend_direction' not in features, "Leakage Alert: trend_direction is in features!"
print("Audit passed: Label-derived features (trend_pct, trend_direction) successfully excluded.\n")

# Generate False Positives / False Negatives using the last fold from the grouped split
preds = clf_g.predict(X_test_g)
results_df = X_test_g.copy()
results_df['Actual'] = y_test_g
results_df['Predicted'] = preds

false_positives = results_df[(results_df['Actual'] == 0) & (results_df['Predicted'] == 1)]
false_negatives = results_df[(results_df['Actual'] == 1) & (results_df['Predicted'] == 0)]

print(f"Found {len(false_positives)} False Positives in the final fold.")
print(f"Found {len(false_negatives)} False Negatives in the final fold.")

print("\nExample False Positive (Model thought it was declining, but it wasn't):")
display(false_positives.head(1))

print("\nExample False Negative (Model thought it was stable, but it declined):")
display(false_negatives.head(1))


: 

## 4. Claim rewrite

**Original bold claim:** 
"My model perfectly predicts exactly which content will fail next month, allowing us to automatically delete bad content and save thousands of dollars."

**Rewritten safe claim:**
"Based on historical data, we **measured** engagement patterns and **observed** a **directional** indicator of content performance decay. This model serves as a **decision-support** tool to help content teams prioritize their refresh pipeline."

In [ ]:
print("Claim successfully rewritten with safe vocabulary.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.